# ChuckleNet: Self-Contained Extraction + Training
## No Google Drive - Data from Kaggle directly

**This notebook:**
1. Downloads WavLM embeddings from Kaggle
2. Downloads utterances with labels from Kaggle
3. Extracts prosody from audio
4. Trains fusion model

**Runtime:** ~45-60 min on T4 GPU

In [ ]:
# @title Step 1: Install dependencies
!pip install -q kaggle librosa scikit-learn

In [ ]:
# @title Step 2: Download data from Kaggle
import os
DATA_DIR = '/content/data'
os.makedirs(DATA_DIR, exist_ok=True)
os.chdir(DATA_DIR)

# Download WavLM embeddings
!kaggle datasets download -d subhajitdas/chuckle-wavlm-555-videos -p {DATA_DIR} --unzip -q

# Download utterances with labels
!kaggle datasets download -d subhajitdas/chuckle-vtt-labels -p {DATA_DIR} --unzip -q

# Download audio (if needed for prosody)
# !kaggle datasets download -d subhajitdas/chuckle-vtt-audio-tar -p {DATA_DIR} --unzip

print('✅ Data downloaded')
!ls -la

In [ ]:
# @title Step 3: Load WavLM embeddings
import json
from pathlib import Path

WAVLM_DIR = Path(f'{DATA_DIR}/chuckle-wavlm-555-videos')

wavlm_data = {}
for json_file in WAVLM_DIR.glob('*.json'):
    vid = json_file.stem
    with open(json_file) as f:
        data = json.load(f)
    wavlm_data[vid] = data['embeddings']

print(f'✅ Loaded WavLM for {len(wavlm_data)} videos')
total = sum(len(v) for v in wavlm_data.values())
print(f'   Total utterances: {total}')

In [ ]:
# @title Step 4: Load utterances with labels
from pathlib import Path

# Find utterances file
UTT_DIR = Path(f'{DATA_DIR}')
utt_files = list(UTT_DIR.rglob('*utterances*.jsonl'))
print(f'Looking for utterances files in {UTT_DIR}')
print(f'Found: {utt_files}')

# Load labels
label_lookup = {}
for ufile in utt_files:
    print(f'Loading from {ufile}...')
    with open(ufile) as f:
        for line in f:
            try:
                u = json.loads(line.strip())
                # Key by video_id + start
                key = (u['video_id'], round(u['start'], 2), round(u['end'], 2))
                label_lookup[key] = u.get('label', 0)
            except:
                pass

print(f'✅ Loaded {len(label_lookup)} labeled utterances')

In [ ]:
# @title Step 5: Match WavLM with labels
matched = 0
unmatched = 0

for vid, embeddings in wavlm_data.items():
    for emb in embeddings:
        key = (vid, round(emb['start'], 2), round(emb['end'], 2))
        if key in label_lookup:
            emb['label'] = label_lookup[key]
            matched += 1
        else:
            emb['label'] = 0  # Default to no laughter
            unmatched += 1

pos = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 1)
neg = sum(1 for v in wavlm_data.values() for e in v if e.get('label') == 0)

print(f'✅ Matched: {matched} | Unmatched: {unmatched}')
print(f'   Positive: {pos} ({pos/(pos+neg)*100:.1f}%)')
print(f'   Negative: {neg} ({neg/(pos+neg)*100:.1f}%)')

In [ ]:
# @title Step 6: Extract prosody (21-dim)
import numpy as np
import librosa
import time
from pathlib import Path

SR = 16000
AUDIO_DIR = Path(f'{DATA_DIR}/chuckle-vtt-audio-tar')

def extract_prosody_21dim(y, sr):
    features = []
    
    # F0 (pitch) - 5 dims
    try:
        f0, voiced_flag, voiced_probs = librosa.pyin(y, fmin=50, fmax=500, sr=sr)
        f0_clean = f0[~np.isnan(f0)]
        features.extend([
            np.mean(f0_clean) if len(f0_clean) > 0 else 0,
            np.std(f0_clean) if len(f0_clean) > 0 else 0,
            np.max(f0_clean) if len(f0_clean) > 0 else 0,
            np.min(f0_clean) if len(f0_clean) > 0 else 0,
            np.sum(voiced_flag) / len(voiced_flag) if len(voiced_flag) > 0 else 0
        ])
    except:
        features.extend([0]*5)
    
    # Energy - 5 dims
    rms = librosa.feature.rms(y=y)[0]
    features.extend([np.mean(rms), np.std(rms), np.max(rms), np.min(rms), np.max(rms) - np.min(rms)])
    
    # Duration - 2 dims
    features.extend([len(y) / sr, len(y) / sr / (np.sum(rms > np.mean(rms)) + 1)])
    
    # Spectral - 5 dims
    spec_cent = librosa.feature.spectral_centroid(y=y, sr=sr)[0]
    spec_bw = librosa.feature.spectral_bandwidth(y=y, sr=sr)[0]
    spec_flat = librosa.feature.spectral_flatness(y=y)[0]
    zcr = librosa.feature.zero_crossing_rate(y)[0]
    features.extend([np.mean(spec_cent), np.mean(spec_bw), np.mean(spec_flat), np.mean(zcr), np.std(zcr)])
    
    # Voice quality - 4 dims
    try:
        hnr = librosa.effects.hpss(y)[1]
        hnr_val = np.mean(hnr) / (np.mean(np.abs(y)) + 1e-8)
    except:
        hnr_val = 0
    features.extend([hnr_val, np.mean(np.abs(y)), np.std(y), np.max(np.abs(y))])
    
    return np.array(features, dtype=np.float32)

def get_audio_path(vid):
    for ext in ['.wav', '.mp3', '.m4a']:
        for match in AUDIO_DIR.rglob(f'{vid}{ext}'):
            return str(match)
        p = AUDIO_DIR / f'{vid}{ext}'
        if p.exists():
            return str(p)
    return None

print('Extracting prosody for all videos...')
t0 = time.time()

prosody_data = {}
failed = []

for i, (vid, embeddings) in enumerate(wavlm_data.items()):
    audio_path = get_audio_path(vid)
    
    if not audio_path:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(embeddings)
        continue
    
    try:
        y, sr = librosa.load(audio_path, sr=SR, mono=True)
        if len(y.shape) > 1:
            y = y.mean(axis=1)
        
        video_prosody = []
        for emb in embeddings:
            start_sample = int(emb['start'] * SR)
            end_sample = int(emb['end'] * SR)
            if end_sample > len(y):
                end_sample = len(y)
            y_slice = y[start_sample:end_sample]
            
            if len(y_slice) < SR * 0.1:
                video_prosody.append(np.zeros(21, dtype=np.float32))
            else:
                prosody = extract_prosody_21dim(y_slice, SR)
                video_prosody.append(prosody)
        
        prosody_data[vid] = video_prosody
        
    except Exception as e:
        failed.append(vid)
        prosody_data[vid] = [np.zeros(21, dtype=np.float32).tolist()] * len(embeddings)
    
    if (i + 1) % 50 == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i + 1) * (len(wavlm_data) - i - 1)
        print(f'{i+1}/{len(wavlm_data)} | ETA: {eta/60:.1f}min | Failed: {len(failed)}')

print(f'\n✅ Done! Prosody: {len(prosody_data)} | Failed: {len(failed)}')
print(f'Time: {(time.time()-t0)/60:.1f} min')

In [ ]:
# @title Step 7: Prepare combined data
import numpy as np

all_emb = []
all_pros = []
all_labels = []

for vid, embeddings in wavlm_data.items():
    if vid not in prosody_data:
        continue
    for i, emb in enumerate(embeddings):
        all_emb.append(emb['embedding'])
        all_pros.append(prosody_data[vid][i])
        all_labels.append(emb.get('label', 0))

all_emb = np.array(all_emb, dtype=np.float32)
all_pros = np.array(all_pros, dtype=np.float32)
all_labels = np.array(all_labels, dtype=np.int64)

print(f'Total: {len(all_emb)} samples')
if len(all_emb) > 0:
    pos = sum(all_labels)
    print(f'Positive: {pos} ({pos/len(all_labels)*100:.1f}%)')
    print(f'WavLM dim: {all_emb.shape[1]}')
    print(f'Prosody dim: {all_pros.shape[1]}')

In [ ]:
# @title Step 8: Train/Val/Test split + Training
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, classification_report
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Split by video (to avoid leakage)
video_ids = list(wavlm_data.keys())
np.random.seed(42)
np.random.shuffle(video_ids)

n = len(video_ids)
val_vids = set(video_ids[:int(n*0.1)])
test_vids = set(video_ids[int(n*0.1):int(n*0.2)])

train_emb, val_emb, train_pros, val_pros, train_lbl, val_lbl = [], [], [], [], [], []
test_emb, test_pros, test_lbl = [], [], []

for vid in video_ids:
    for i, emb in enumerate(wavlm_data[vid]):
        if vid not in prosody_data:
            continue
        row = (all_emb if vid in [v for v in wavlm_data.keys()][:1] else None)  # placeholder
        
        # Find matching index in all_emb
        for j, (e, p, l) in enumerate(zip(all_emb, all_pros, all_labels)):
            if abs(e - emb['embedding']).sum() < 0.01:  # Match by embedding
                if vid in val_vids:
                    val_emb.append(all_emb[j])
                    val_pros.append(all_pros[j])
                    val_lbl.append(all_labels[j])
                elif vid in test_vids:
                    test_emb.append(all_emb[j])
                    test_pros.append(all_pros[j])
                    test_lbl.append(all_labels[j])
                else:
                    train_emb.append(all_emb[j])
                    train_pros.append(all_pros[j])
                    train_lbl.append(all_labels[j])
                break

train_emb = np.array(train_emb, dtype=np.float32)
val_emb = np.array(val_emb, dtype=np.float32)
test_emb = np.array(test_emb, dtype=np.float32)
train_pros = np.array(train_pros, dtype=np.float32)
val_pros = np.array(val_pros, dtype=np.float32)
test_pros = np.array(test_pros, dtype=np.float32)
train_lbl = np.array(train_lbl, dtype=np.int64)
val_lbl = np.array(val_lbl, dtype=np.int64)
test_lbl = np.array(test_lbl, dtype=np.int64)

print(f'Train: {len(train_emb)} | Val: {len(val_emb)} | Test: {len(test_emb)}')

In [ ]:
# @title Step 9: Create DataLoaders
train_ds = TensorDataset(
    torch.tensor(train_emb, dtype=torch.float32),
    torch.tensor(train_pros, dtype=torch.float32),
    torch.tensor(train_lbl, dtype=torch.long)
)
val_ds = TensorDataset(
    torch.tensor(val_emb, dtype=torch.float32),
    torch.tensor(val_pros, dtype=torch.float32),
    torch.tensor(val_lbl, dtype=torch.long)
)
test_ds = TensorDataset(
    torch.tensor(test_emb, dtype=torch.float32),
    torch.tensor(test_pros, dtype=torch.float32),
    torch.tensor(test_lbl, dtype=torch.long)
)

train_loader = DataLoader(train_ds, batch_size=256, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=256)
test_loader = DataLoader(test_ds, batch_size=256)

print(f'Train batches: {len(train_loader)}')

In [ ]:
# @title Step 10: Model + Training (ALL FIXES)
class FusionModel(nn.Module):
    def __init__(self, wavlm_dim=768, prosody_dim=21):
        super().__init__()
        self.prosody_proj = nn.Sequential(
            nn.Linear(prosody_dim, 64),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(wavlm_dim + 32, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.GELU(),
            nn.Linear(64, 2)
        )
    
    def forward(self, wavlm_emb, prosody):
        prosody_feat = self.prosody_proj(prosody)
        x = torch.cat([wavlm_emb, prosody_feat], dim=-1)
        return self.classifier(x)

model = FusionModel().to(device)

# FIXED: Class weights [1.0, 2.5] + CrossEntropyLoss
class_weights = torch.tensor([1.0, 2.5], dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20)

EPOCHS = 20
best_f1 = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    
    # Train
    model.train()
    train_loss = 0
    for emb_b, pros_b, labels_b in train_loader:
        emb_b = emb_b.to(device)
        pros_b = pros_b.to(device)
        labels_b = labels_b.to(device)
        
        optimizer.zero_grad()
        logits = model(emb_b, pros_b)
        loss = criterion(logits, labels_b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    
    scheduler.step()
    
    # Validate
    model.eval()
    val_preds, val_labels = [], []
    with torch.no_grad():
        for emb_b, pros_b, labels_b in val_loader:
            emb_b = emb_b.to(device)
            pros_b = pros_b.to(device)
            logits = model(emb_b, pros_b)
            preds = torch.argmax(logits, dim=-1)
            val_preds.extend(preds.cpu().numpy())
            val_labels.extend(labels_b.numpy())
    
    val_f1 = f1_score(val_labels, val_preds, average='binary')
    epoch_time = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss/len(train_loader):.4f} | Val F1: {val_f1:.4f} | Time: {epoch_time:.1f}s')
    
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, '/content/best_model.pt')
        print(f'  ✅ New best!')

In [ ]:
# @title Step 11: Final Evaluation
model.load_state_dict(best_state)
model.eval()

test_preds, test_labels = [], []
with torch.no_grad():
    for emb_b, pros_b, labels_b in test_loader:
        emb_b = emb_b.to(device)
        pros_b = pros_b.to(device)
        logits = model(emb_b, pros_b)
        preds = torch.argmax(logits, dim=-1)
        test_preds.extend(preds.cpu().numpy())
        test_labels.extend(labels_b.numpy())

test_f1 = f1_score(test_labels, test_preds, average='binary')
print(f'\n🏆 Test F1: {test_f1:.4f}')
print(classification_report(test_labels, test_preds, target_names=['No Laughter', 'Laughter']))

# Save
results = {
    'test_f1': float(test_f1),
    'val_f1': float(best_f1),
    'n_train': len(train_emb),
    'n_val': len(val_emb),
    'n_test': len(test_emb),
    'n_failed_audio': len(failed)
}
with open('/content/results.json', 'w') as f:
    json.dump(results, f, indent=2)

print('\n✅ Done! Model saved to /content/best_model.pt')